In [1]:
# Import libraries
import os
import sys
import boto3

from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from lab_helpers.utils import (
    get_or_create_cognito_pool,
    put_ssm_parameter,
    get_ssm_parameter,
    load_api_spec,
)


sts_client = boto3.client("sts")

# Get AWS account details
REGION = boto3.session.Session().region_name

gateway_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=REGION,
)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [2]:
gateway_name = "customersupport-gw"

cognito_config = get_or_create_cognito_pool(refresh_token=True)
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [cognito_config["client_id"]],
        "discoveryUrl": cognito_config["discovery_url"],
    }
}

try:
    # create new gateway
    print(f"Creating gateway in region {REGION} with name: {gateway_name}")

    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn=get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role"),
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration=auth_config,
        description="Customer Support AgentCore Gateway",
    )

    gateway_id = create_response["gatewayId"]

    gateway = {
        "id": gateway_id,
        "name": gateway_name,
        "gateway_url": create_response["gatewayUrl"],
        "gateway_arn": create_response["gatewayArn"],
    }
    put_ssm_parameter("/app/customersupport/agentcore/gateway_id", gateway_id)
    put_ssm_parameter("/app/customersupport/agentcore/gateway_name", gateway_name)
    put_ssm_parameter(
        "/app/customersupport/agentcore/gateway_arn", create_response["gatewayArn"]
    )
    put_ssm_parameter(
        "/app/customersupport/agentcore/gateway_url", create_response["gatewayUrl"]
    )
    print(f"✅ Gateway created successfully with ID: {gateway_id}")

except Exception:
    # If gateway exists, collect existing gateway ID from SSM
    existing_gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    print(f"Found existing gateway with ID: {existing_gateway_id}")

    # Get existing gateway details
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=existing_gateway_id)
    gateway = {
        "id": existing_gateway_id,
        "name": gateway_response["name"],
        "gateway_url": gateway_response["gatewayUrl"],
        "gateway_arn": gateway_response["gatewayArn"],
    }
    gateway_id = gateway["id"]

❌ Error getting secret: An error occurred (ResourceNotFoundException) when calling the GetSecretValue operation: Secrets Manager can't find the specified secret.
No existing cognito config found. Creating a new one..
{'UserPoolId': 'us-west-2_0vBmv65w4', 'ClientName': 'MCPServerPoolClient', 'ClientId': '4iriigpq6h9t33nqmpoedk59j6', 'ClientSecret': '1beiakcbknqv7enuos3t25sbenqhrass333eihgg7c05icom5603', 'LastModifiedDate': datetime.datetime(2026, 10, 20, 4, 19, 12, 622000, tzinfo=tzlocal()), 'CreationDate': datetime.datetime(2026, 10, 20, 4, 19, 12, 622000, tzinfo=tzlocal()), 'RefreshTokenValidity': 30, 'TokenValidityUnits': {}, 'ExplicitAuthFlows': ['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_USER_SRP_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH'], 'AllowedOAuthFlowsUserPoolClient': False, 'EnableTokenRevocation': True, 'EnablePropagateAdditionalUserContextData': False, 'AuthSessionValidity': 3}
Pool id: us-west-2_0vBmv65w4
Discovery URL: https://cognito-idp.us-west-2.amazonaws.com/us-west-2_0vBmv65w4/.wel

In [3]:
try:
    api_spec_file = "./prerequisite/lambda/api_spec.json"

    # Validate API spec file exists
    if not os.path.exists(api_spec_file):
        print(f"❌ API specification file not found: {api_spec_file}")
        sys.exit(1)

    api_spec = load_api_spec(api_spec_file)

    # Use Cognito for Inbound OAuth to our Gateway
    lambda_target_config = {
        "mcp": {
            "lambda": {
                "lambdaArn": get_ssm_parameter(
                    "/app/customersupport/agentcore/lambda_arn"
                ),
                "toolSchema": {"inlinePayload": api_spec},
            }
        }
    }

    # Create gateway target
    credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]

    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="LambdaUsingSDK",
        description="Lambda Target using SDK",
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=credential_config,
    )

    print(f"✅ Gateway target created: {create_target_response['targetId']}")

except Exception as e:
    print(f"❌ Error creating gateway target: {str(e)}")

✅ Gateway target created: WQHAHF4NF3


In [4]:
print(f"Gateway Endpoint - MCP URL: {gateway['gateway_url']}")
# Set up MCP client
mcp_client = MCPClient(
    lambda: streamablehttp_client(
        gateway["gateway_url"],
        headers={"Authorization": f"Bearer {cognito_config['bearer_token']}"},
    )
)

Gateway Endpoint - MCP URL: https://customersupport-gw-lu2dumyajx.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp


In [5]:
from lab_helpers.lab1_strands_agent import (
    get_product_info,
    get_return_policy,
    get_technical_support,
    SYSTEM_PROMPT,
)
from lab_helpers.lab2_memory import (
    CustomerSupportMemoryHooks,
    create_or_get_memory_resource,
)
import uuid
from bedrock_agentcore.memory import MemoryClient

memory_client = MemoryClient(region_name=REGION)

memory_id = create_or_get_memory_resource()
SESSION_ID = str(uuid.uuid4())
CUSTOMER_ID = "customer_001"
memory_hooks = CustomerSupportMemoryHooks(
    memory_id, memory_client, CUSTOMER_ID, SESSION_ID
)

# Initialize the Bedrock model
model_id = "us.amazon.nova-2-lite-v1:0"
model = BedrockModel(
    model_id=model_id,
    temperature=0.3,  # Balanced between creativity and consistency
    region_name=REGION,
)


def create_agent(prompt):
    try:
        with mcp_client:
            tools = [
                get_product_info,
                get_return_policy,
                get_technical_support,
            ] + mcp_client.list_tools_sync()

            # Create the customer support agent
            agent = Agent(
                model=model,
                tools=tools,
                hooks=[memory_hooks],
                system_prompt=SYSTEM_PROMPT,
            )
            response = agent(prompt)
            return response
    except Exception as e:
        raise e


print("✅ Customer support agent created successfully!")

✅ Customer support agent created successfully!


In [6]:
# test 

test_prompts = [
    # Warranty Checks
    "List all of your tools",
    "I bought an iphone 14 last month. I don't like it because it heats up. How do I solve it?",
    "I have a Gaming Console Pro device , I want to check my warranty status, warranty serial number is MNO33333333.",
    "What are the warranty support guidelines?",
    "How can I fix Lenovo Thinkpad with a blue screen",
    "Tell me detailed information about the technical documentation on installing a new CPU",
]


# Function to test the agent
def test_agent_responses(prompts):
    for i, prompt in enumerate(prompts, 1):
        print(f"\nTest Case {i}: {prompt}")
        print("-" * 50)
        try:
            response = create_agent(prompt)
            print(response)
        except Exception as e:
            print(f"Error: {str(e)}")
        print("-" * 50)


# Run the tests
test_agent_responses(test_prompts)

print("\\n✅ Basic testing completed!")


Test Case 1: List all of your tools
--------------------------------------------------
I can see you've shared several user contexts and preferences. Let me help you with the appropriate tools based on your needs.

### Available Tools:

1. **`get_product_info(product_type)`** - Get detailed technical specifications and information for electronics products
   - Perfect for checking gaming headphones with low latency audio for FPS games
   - Also useful for laptop information (especially ThinkPad models for programming)

2. **`get_return_policy(product_category)`** - Get return policy information for specific product categories
   - Useful if you need to understand return options for any products

3. **`get_technical_support(issue_description)`** - Get technical support for specific issues
   - Ideal for addressing your MacBook Pro overheating during video editing

4. **`LambdaUsingSDK___check_warranty_status(serial_number, customer_email)`** - Check warranty status of products
   - Can

DEPRECATION WARNING: retrieve is deprecated. This warning becomes an error log in v0.9.0. Migration path: use MemoryManager with BedrockKnowledgeBaseStore(writable=False). Note the store applies no relevance-score floor, so filter results yourself. See https://strandsagents.com/docs/user-guide/concepts/memory/overview/



Tool #1: get_product_info

Tool #2: get_technical_support
Successfully retrieved KB ID: AA3OGUUIZB
Based on the technical support information I retrieved, here's how you can address the blue screen issue on your Lenovo ThinkPad:

## Understanding Blue Screen Errors

A blue screen error (also known as a Blue Screen of Death or BSOD) indicates a critical system error that's causing your Windows operating system to crash. These errors can be caused by various factors including hardware issues, driver conflicts, or software problems.

## Step-by-Step Troubleshooting for Your Lenovo ThinkPad

### 1. **Immediate Actions**
- **Don't restart immediately** - Note any error codes or messages displayed on the blue screen. These codes (like IRQL_NOT_LESS_OR_EQUAL or PAGE_FAULT_IN_NONPAGED_AREA) can help identify the specific issue.
- **If it happens repeatedly**, try booting into **Safe Mode** (press F8 or Shift+Restart during startup) to see if the issue persists with minimal drivers loaded.

##